# One-liner SED fitting with tengri

This notebook demonstrates the minimal API for SED fitting with tengri. Start here to understand the core workflow: load data, choose a model preset, fit, and explain results.

**Note:** This notebook requires access to SSP (Single Stellar Population) data. Set the environment variable `TENGRI_SSP_PATH` to point to your SSP data directory, or pass `ssp_path=...` explicitly to `Galaxy.from_arrays()`. If SSP data is not available, fitting will be skipped.

In [1]:
import tengri as tg

tg.print_logo()
print(f"tengri {tg.__version__}")

tg.doctor()

W0423 18:24:09.258331 24171805 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


                            ████████
                        ████████████████
                     ██████          ██████
                 ███████                ███████
              ██████                        ██████
           ██████                              ██████
        ██████         █████████████████          ██████
     ██████        ██████            ███████         ██████
  █████         █████                     █████          █████
 ████         ████        ██████████         ████          ████
████        ████     ████████   ████████       ████         ████
███        ███    ████       ████     █████      ████        ███
███       ███  ████     ██████████████   ████      ███       ███
███      ███ ███     █████         █████   ███      ███      ███
███     ███ ██     ████    ████████   ████  ███      ███     ███
███     ██ ██     ███   ██████  █████  ████  ███      ███    ███
███    ██ █      ███  ████  ██████ ███  ███  ███      ███    ███
███    ████     ███  ███  

tengri 0.1.0
                            ████████
                        ████████████████
                     ██████          ██████
                 ███████                ███████
              ██████                        ██████
           ██████                              ██████
        ██████         █████████████████          ██████
     ██████        ██████            ███████         ██████
  █████         █████                     █████          █████
 ████         ████        ██████████         ████          ████
████        ████     ████████   ████████       ████         ████
███        ███    ████       ████     █████      ████        ███
███       ███  ████     ██████████████   ████      ███       ███
███      ███ ███     █████         █████   ███      ███      ███
███     ███ ██     ████    ████████   ████  ███      ███     ███
███     ██ ██     ███   ██████  █████  ████  ███      ███    ███
███    ██ █      ███  ████  ██████ ███  ███  ███      ███    ███
███    ████  

'                            ████████\n                        ████████████████\n                     ██████          ██████\n                 ███████                ███████\n              ██████                        ██████\n           ██████                              ██████\n        ██████         █████████████████          ██████\n     ██████        ██████            ███████         ██████\n  █████         █████                     █████          █████\n ████         ████        ██████████         ████          ████\n████        ████     ████████   ████████       ████         ████\n███        ███    ████       ████     █████      ████        ███\n███       ███  ████     ██████████████   ████      ███       ███\n███      ███ ███     █████         █████   ███      ███      ███\n███     ███ ██     ████    ████████   ████  ███      ███     ███\n███     ██ ██     ███   ██████  █████  ████  ███      ███    ███\n███    ██ █      ███  ████  ██████ ███  ███  ███      ███    ███\n███    █

## See which models are available

In [2]:
tg.presets.list_presets()
print(tg.presets.describe("starforming"))

Starforming galaxies (main sequence, z ~ 0–3):
- SFH: double power-law (α, β, τ, log(SFR_peak))
- Dust: Calzetti (birth cloud) + power-law (diffuse ISM), two-component
- Av: Uniform(0, 2) mag
- Metallicity: Uniform(−0.5, +0.3) [Z/Zsun]
- Nebular: off (baked-in SSP default)
- Redshift: free Uniform(0.01, 6.0) unless specified

Use for: optical/NIR surveys, moderate-redshift galaxies with ongoing star formation.



## Inspect the citations you'll pick up by using tengri

In [3]:
for c in tg.cite_all():
    print(c)

[Reference SED fitting framework (BAGPIPES)] — Carnall et al. (2018). DOI: 10.1093/mnras/sty1931. (upstream: ACCarnall/bagpipes)
[MCMC inference (NUTS sampler)] — Cabezas et al. (2023). arXiv: 2402.00787. (upstream: blackjax-devs/blackjax)
[Starburst dust attenuation law] — Calzetti et al. (2000). DOI: 10.1086/308692
[Two-component dust attenuation (BC+ISM)] — Charlot & Fall (2000). DOI: 10.1086/309250
[Nebular emission-line emulator] — Li et al. (2024). arXiv: 2405.07657. (upstream: yi-jia-li/cue)
[Differentiable stellar population synthesis engine] — Hearin et al. (2023). DOI: 10.1093/mnras/stad1905. (upstream: ArgonneCPAC/dsps)
[3D Galactic dust map (MW extinction preprocessing)] — Edenhofer et al. (2023). DOI: 10.1051/0004-6361/202346487
[Stellar population synthesis (reference implementation)] — Conroy & Gunn (2010). DOI: 10.1088/0004-637X/712/2/833. (upstream: cconroy20/fsps)
[IGM Lyman series attenuation] — Inoue et al. (2014). DOI: 10.1093/mnras/stu1657
[Autodiff framework and 

## Build a Galaxy from arrays (demo data; replace with your own)

In [4]:
import os

ssp_path = os.environ.get("TENGRI_SSP_PATH")
if ssp_path and os.path.exists(ssp_path):
    g = tg.Galaxy.from_arrays(
        filters=["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"],
        flux=[1.0e-28, 2.0e-28, 3.0e-28, 2.5e-28, 2.0e-28],
        flux_err=[1.0e-29] * 5,
        flux_unit="erg/s/cm2/Hz",
        redshift=0.1,
        ssp_path=ssp_path,
        preset="starforming",
    )
    g.fit(backend="map")
    print(g.summary())
    print(g.explain())
else:
    print("Set TENGRI_SSP_PATH to run a real fit. Skipping.")

Set TENGRI_SSP_PATH to run a real fit. Skipping.


## That's the whole flow: load, fit, summarise, cite.

For more details, see the documentation and other notebooks in this directory.